# Custom Comparison Dashboard

Edit the selection cell, then run the notebook. It compares any mix of:

- W&B cached training histories selected by config folder, exact run name, or run-name substring/regex,
- standalone full-test JSON summaries,
- teacher-student dataset metadata exports,
- behavior-cloned student checkpoints.

This notebook is intentionally generic: write the config folder name, run name, checkpoint name, dataset name, or a substring in the selection cell and the remaining cells will filter everything that matches.


In [ ]:
from pathlib import Path
import importlib
import re
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "comparison_dashboard.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "comparison_dashboard.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import comparison_dashboard as cd
cd = importlib.reload(cd)
print("comparison_dashboard:", cd.__file__)
print("TASK_DIR:", cd.TASK_DIR)


## Selection

This is the only cell you normally edit.

Examples:

```python
CONFIG_FOLDERS = ["a0_hvg"]
QUERY_NAMES = ["a0_hvg_04_eval_local_rho090_s0", "local_bc_s0"]
```

`QUERY_NAMES` are substring filters applied to W&B run names, full-test checkpoint names, teacher dataset names, and student checkpoint names.


In [ ]:
# Config folders under Topology_Task/configs. Leave empty if you only want name-based selection.
CONFIG_FOLDERS = []

# Exact W&B run names, if you know them exactly. Usually optional.
EXACT_RUN_NAMES = []

# Substrings applied everywhere: W&B run names, full-test JSON names, dataset names, and student checkpoint names.
# Put the names you care about here.
QUERY_NAMES = [
    # "a0_hvg_04_eval_local_rho090_s0",
    # "local_bc_s0",
]

# Optional raw regexes for each source. Use these when substrings are not enough.
RUN_NAME_REGEX = ""
FULL_TEST_REGEX = ""
TEACHER_DATASET_REGEX = ""
STUDENT_CHECKPOINT_REGEX = ""

# If everything above is empty, this controls whether the notebook loads all local cached W&B histories.
LOAD_ALL_WANDB_IF_EMPTY = False
SHOW_ALL_FULL_TEST_IF_EMPTY = False
SHOW_ALL_TEACHER_DATASETS_IF_EMPTY = False
SHOW_ALL_STUDENT_CHECKPOINTS_IF_EMPTY = False

SMOOTH = 3
FINAL_LAST_N = 5
STEP_MAX_M = None
SHOW_FIGURES = True
SAVE_FIGURES = True

SELECTION_LABEL = cd.safe_name("_".join(CONFIG_FOLDERS + EXACT_RUN_NAMES + QUERY_NAMES) or "custom")
print("selection label:", SELECTION_LABEL)


## Helper Functions


In [ ]:
def _as_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value or value.lower() in {"none", "null"}:
            return []
        return [part.strip() for part in value.split(",") if part.strip()]
    return [str(item).strip() for item in value if str(item).strip()]


def _combined_regex(substrings=None, raw_regex=""):
    parts = [re.escape(item) for item in _as_list(substrings)]
    raw_regex = str(raw_regex or "").strip()
    if raw_regex:
        parts.append(f"(?:{raw_regex})")
    return "|".join(parts)


def _filter_by_any_text(df, columns, *, substrings=None, raw_regex="", keep_all_if_empty=False):
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()
    pattern = _combined_regex(substrings, raw_regex)
    if not pattern:
        return df.copy() if keep_all_if_empty else df.iloc[0:0].copy()
    mask = pd.Series(False, index=df.index)
    for column in columns:
        if column in df.columns:
            mask = mask | df[column].astype(str).str.contains(pattern, case=False, regex=True, na=False)
    return df[mask].copy()


def _show(df, columns=None, sort=None, n=200):
    if df is None or df.empty:
        display(pd.DataFrame())
        return
    out = df.copy()
    if sort:
        sort_cols = [c for c in sort if c in out.columns]
        if sort_cols:
            out = out.sort_values(sort_cols)
    if columns:
        cols = [c for c in columns if c in out.columns]
        out = out[cols]
    display(out.head(n))


def _plot_and_final(curve, *, title, y_title, save_name, facet_col=None, extra_group_cols=()):
    if curve.empty:
        print(f"No data: {title}")
        return pd.DataFrame()
    cd.plot_timeseries(
        curve,
        title=title,
        y_title=y_title,
        facet_col=facet_col,
        extra_group_cols=extra_group_cols,
        save_name=f"{SELECTION_LABEL}_{save_name}_timeseries",
        show=SHOW_FIGURES,
        save=SAVE_FIGURES,
    )
    final = cd.per_run_final(
        curve,
        last_n=FINAL_LAST_N,
        group_cols=("run_id", *tuple(extra_group_cols)),
    )
    cd.plot_final_bar(
        final,
        title=f"{title}, final window",
        y_title=y_title,
        extra_group_cols=extra_group_cols,
        save_name=f"{SELECTION_LABEL}_{save_name}_final",
        show=SHOW_FIGURES,
        save=SAVE_FIGURES,
    )
    return final


## Select And Load W&B Histories


In [ ]:
config_folders = _as_list(CONFIG_FOLDERS)
exact_run_names = _as_list(EXACT_RUN_NAMES)
run_regex = _combined_regex(QUERY_NAMES, RUN_NAME_REGEX)

if config_folders or exact_run_names or run_regex:
    selected_runs = cd.select_cached_runs(
        config_folders=config_folders or None,
        exact_run_names=exact_run_names or None,
        regex=run_regex or None,
    )
elif LOAD_ALL_WANDB_IF_EMPTY:
    selected_runs = cd.select_cached_runs()
else:
    selected_runs = pd.DataFrame()

print("selected W&B runs:", len(selected_runs))
_show(
    selected_runs,
    columns=[
        "run_name", "condition", "seed", "experiment", "mechanism", "run_id", "rows", "state"
    ],
    sort=["condition_order", "seed", "run_name"],
)

if not selected_runs.empty:
    history_df = cd.load_cached_histories(selected_runs, step_max_m=STEP_MAX_M, verbose=True)
else:
    history_df = pd.DataFrame()
print("history rows:", len(history_df))


## Metric Availability


In [ ]:
if history_df.empty:
    print("No W&B history loaded. Fill CONFIG_FOLDERS, EXACT_RUN_NAMES, or QUERY_NAMES above.")
    availability = pd.DataFrame()
else:
    availability = cd.metric_availability(history_df)
_show(availability, sort=["condition", "seed", "run_name"])


## Survival Curves


In [ ]:
survival_finals = []
for split in ["test", "train_eval", "train"]:
    curve = cd.survival_curve(history_df, split=split, smooth=SMOOTH) if not history_df.empty else pd.DataFrame()
    final = _plot_and_final(
        curve,
        title=f"{split} episodic survival",
        y_title="survival",
        save_name=f"{split}_survival",
    )
    if not final.empty:
        final["split"] = split
        survival_finals.append(final)

survival_final_df = pd.concat(survival_finals, ignore_index=True, sort=False) if survival_finals else pd.DataFrame()
_show(
    survival_final_df,
    columns=["split", "run_name", "condition", "seed", "final_value", "final_step_millions", "n_logged_points"],
    sort=["split", "condition_order", "seed"],
)


## Action-0, Non-Idle, And Illegal-Action Curves


In [ ]:
agent_final_tables = []
agent_specs = [
    ("action0", "eval", "test", "test action-0 fraction", "test_action0", "action 0 fraction"),
    ("action0", "train", "test", "train action-0 fraction", "train_action0", "action 0 fraction"),
    ("nonidle", "eval", "test", "test non-idle fraction", "test_nonidle", "non-idle fraction"),
    ("nonidle", "train", "test", "train non-idle fraction", "train_nonidle", "non-idle fraction"),
    ("illegal", "train", "test", "train illegal action rate", "train_illegal", "illegal action rate"),
]
for metric, source, split, title, save_name, y_title in agent_specs:
    curve = (
        cd.agent_metric_curve(history_df, metric=metric, source=source, split=split, smooth=SMOOTH)
        if not history_df.empty else pd.DataFrame()
    )
    final = _plot_and_final(
        curve,
        title=title,
        y_title=y_title,
        save_name=save_name,
        facet_col="agent",
        extra_group_cols=("agent",),
    )
    if not final.empty:
        final["metric_kind"] = save_name
        agent_final_tables.append(final)

agent_final_df = pd.concat(agent_final_tables, ignore_index=True, sort=False) if agent_final_tables else pd.DataFrame()
_show(
    agent_final_df,
    columns=["metric_kind", "agent", "run_name", "condition", "seed", "final_value", "final_step_millions"],
    sort=["metric_kind", "agent", "condition_order", "seed"],
)


## Joint Non-Idle Curves


In [ ]:
joint_curve = cd.joint_nonidle_curve(history_df, smooth=SMOOTH) if not history_df.empty else pd.DataFrame()
joint_final = _plot_and_final(
    joint_curve,
    title="joint non-idle agents per environment step",
    y_title="fraction / mean",
    save_name="joint_nonidle",
    facet_col="non_idle_agents",
    extra_group_cols=("non_idle_agents",),
)
_show(
    joint_final,
    columns=["non_idle_agents", "run_name", "condition", "seed", "final_value", "final_step_millions"],
    sort=["non_idle_agents", "condition_order", "seed"],
)


## Heuristic Override, Gate, Sparse Penalty, And AIB Diagnostics


In [ ]:
diagnostic_specs = [
    ("heuristic", cd.heuristic_curve(history_df, split="test", smooth=SMOOTH) if not history_df.empty else pd.DataFrame(), "heuristic override diagnostics", "heuristic", "value", ("metric", "agent"), "metric"),
    ("gate_prob", cd.agent_metric_curve(history_df, metric="gate_prob_intervene", smooth=SMOOTH) if not history_df.empty else pd.DataFrame(), "gate P(intervene)", "gate_prob_intervene", "probability", ("agent",), "agent"),
    ("gate_actual", cd.agent_metric_curve(history_df, metric="gate_actual_intervene", smooth=SMOOTH) if not history_df.empty else pd.DataFrame(), "gate actual intervention fraction", "gate_actual_intervene", "fraction", ("agent",), "agent"),
    ("gate_entropy", cd.agent_metric_curve(history_df, metric="gate_entropy", smooth=SMOOTH) if not history_df.empty else pd.DataFrame(), "gate entropy", "gate_entropy", "entropy", ("agent",), "agent"),
    ("sparse", cd.sparse_penalty_curve(history_df, smooth=SMOOTH) if not history_df.empty else pd.DataFrame(), "fixed sparse penalty diagnostics", "sparse_penalty", "value", ("metric", "agent"), "metric"),
    ("aib", cd.aib_curve(history_df, smooth=SMOOTH) if not history_df.empty else pd.DataFrame(), "adaptive intervention budget diagnostics", "aib", "value", ("metric", "agent"), "metric"),
]

diagnostic_finals = []
for kind, curve, title, save_name, y_title, group_cols, facet in diagnostic_specs:
    if curve.empty:
        print(f"No data: {title}")
        continue
    final = _plot_and_final(
        curve,
        title=title,
        y_title=y_title,
        save_name=save_name,
        facet_col=facet,
        extra_group_cols=group_cols,
    )
    if not final.empty:
        final["diagnostic_kind"] = kind
        diagnostic_finals.append(final)

diagnostic_final_df = pd.concat(diagnostic_finals, ignore_index=True, sort=False) if diagnostic_finals else pd.DataFrame()
_show(
    diagnostic_final_df,
    columns=["diagnostic_kind", "metric", "agent", "run_name", "condition", "seed", "final_value", "final_step_millions"],
    sort=["diagnostic_kind", "metric", "agent", "condition_order", "seed"],
)


## Action-0 Versus Survival Tradeoff


In [ ]:
if history_df.empty:
    tradeoff_per_run, tradeoff_agg = pd.DataFrame(), pd.DataFrame()
else:
    tradeoff_fig = cd.plot_action_survival_tradeoff(
        history_df,
        action_source="eval",
        split="test",
        last_n=FINAL_LAST_N,
        save_name=f"{SELECTION_LABEL}_action0_survival_tradeoff",
        show=SHOW_FIGURES,
        save=SAVE_FIGURES,
    )
    tradeoff_per_run, tradeoff_agg = cd.action_survival_tradeoff(
        history_df,
        action_source="eval",
        split="test",
        last_n=FINAL_LAST_N,
    )
_show(tradeoff_agg, sort=["condition_order"])


## Full-Test Evaluation Results


In [ ]:
full_test_all = cd.load_full_test_results(cd.FULL_TEST_DIR)
full_test_filters = QUERY_NAMES
full_test_df = _filter_by_any_text(
    full_test_all,
    ["run_like", "checkpoint_stem", "checkpoint", "path", "eval_action_heuristic"],
    substrings=full_test_filters,
    raw_regex=FULL_TEST_REGEX,
    keep_all_if_empty=SHOW_ALL_FULL_TEST_IF_EMPTY,
)
print("matching full-test JSON rows:", len(full_test_df), "/", len(full_test_all))
_show(
    full_test_df,
    columns=["run_like", "checkpoint_stem", "checkpoint_global_step", "split", "eval_episodes", "eval_action_heuristic", "survival_percent", "obs_normalization", "path"],
    sort=["run_like", "checkpoint_global_step"],
)
cd.plot_full_test_results(full_test_df, show=SHOW_FIGURES, save=SAVE_FIGURES)


## Teacher Dataset Summaries


In [ ]:
dataset_all, dataset_agent_all = cd.load_teacher_dataset_summaries(cd.TEACHER_DATASET_ROOT)
dataset_df = _filter_by_any_text(
    dataset_all,
    ["dataset", "checkpoint", "metadata_path", "eval_action_heuristic"],
    substrings=QUERY_NAMES,
    raw_regex=TEACHER_DATASET_REGEX,
    keep_all_if_empty=SHOW_ALL_TEACHER_DATASETS_IF_EMPTY,
)
if dataset_df.empty:
    dataset_agent_df = dataset_agent_all.iloc[0:0].copy()
else:
    dataset_agent_df = dataset_agent_all[dataset_agent_all["dataset"].isin(dataset_df["dataset"])].copy()

print("matching teacher datasets:", len(dataset_df), "/", len(dataset_all))
_show(
    dataset_df,
    columns=["dataset", "status", "split", "eval_action_heuristic", "eval_action_rho_threshold", "n_env_steps", "n_agent_examples", "n_completed_episodes", "n_unique_chronic_fingerprints", "n_shards"],
    sort=["dataset"],
)
_show(dataset_agent_df, sort=["dataset", "agent"])
cd.plot_teacher_dataset_balance(dataset_agent_df, show=SHOW_FIGURES, save=SAVE_FIGURES)


## Student BC Checkpoints And Training Curves


In [ ]:
student_ckpt_all, student_epoch_all = cd.load_teacher_student_checkpoints(cd.TEACHER_STUDENT_CHECKPOINT_DIR)
student_ckpt_df = _filter_by_any_text(
    student_ckpt_all,
    ["checkpoint", "dataset", "teacher_checkpoint", "student_training_objective", "path"],
    substrings=QUERY_NAMES,
    raw_regex=STUDENT_CHECKPOINT_REGEX,
    keep_all_if_empty=SHOW_ALL_STUDENT_CHECKPOINTS_IF_EMPTY,
)
if student_ckpt_df.empty:
    student_epoch_df = student_epoch_all.iloc[0:0].copy()
else:
    student_epoch_df = student_epoch_all[student_epoch_all["checkpoint"].isin(student_ckpt_df["checkpoint"])].copy()

print("matching student checkpoints:", len(student_ckpt_df), "/", len(student_ckpt_all))
_show(
    student_ckpt_df,
    columns=["checkpoint", "dataset", "global_step", "epochs", "balanced_nonidle_frac", "nonidle_weight", "aux_intervention_loss", "aux_weight", "soft_distillation_loss", "soft_weight", "soft_temperature", "student_training_objective"],
    sort=["checkpoint"],
)
_show(student_epoch_df, sort=["checkpoint", "epoch", "phase", "agent"])
cd.plot_student_bc_metrics(student_epoch_df, show=SHOW_FIGURES, save=SAVE_FIGURES)


## Student Prediction Rate Versus Teacher Dataset Target


In [ ]:
if not student_epoch_df.empty and not dataset_agent_df.empty:
    latest_eval = (
        student_epoch_df[student_epoch_df["phase"].eq("eval")]
        .sort_values(["checkpoint", "agent", "epoch"])
        .groupby(["checkpoint", "agent"], dropna=False, observed=True)
        .tail(1)
        .copy()
    )
    checkpoint_meta_cols = [
        "checkpoint", "dataset", "balanced_nonidle_frac", "nonidle_weight", "aux_weight",
        "soft_distillation_loss", "soft_weight", "soft_temperature", "student_training_objective",
    ]
    checkpoint_meta_cols = [c for c in checkpoint_meta_cols if c in student_ckpt_df.columns]
    checkpoint_meta = student_ckpt_df[checkpoint_meta_cols].copy()
    if "dataset" in checkpoint_meta.columns:
        checkpoint_meta = checkpoint_meta.rename(columns={"dataset": "dataset_from_checkpoint"})
    latest_eval = latest_eval.merge(checkpoint_meta, on="checkpoint", how="left")
    if "dataset" not in latest_eval.columns:
        latest_eval["dataset"] = latest_eval.get("dataset_from_checkpoint")
    elif "dataset_from_checkpoint" in latest_eval.columns:
        latest_eval["dataset"] = latest_eval["dataset"].fillna(latest_eval["dataset_from_checkpoint"])
    latest_eval = latest_eval.drop(columns=["dataset_from_checkpoint"], errors="ignore")

    target = dataset_agent_df[["dataset", "agent"]].copy()
    if "teacher_nonidle_frac" in dataset_agent_df.columns:
        target["target_teacher_nonidle_frac"] = dataset_agent_df["teacher_nonidle_frac"]
    elif "teacher_action0_frac" in dataset_agent_df.columns:
        target["target_teacher_nonidle_frac"] = 1.0 - dataset_agent_df["teacher_action0_frac"]
    if "teacher_action0_frac" in dataset_agent_df.columns:
        target["target_teacher_action0_frac"] = dataset_agent_df["teacher_action0_frac"]
    if "was_overwritten_frac" in dataset_agent_df.columns:
        target["target_was_overwritten_frac"] = dataset_agent_df["was_overwritten_frac"]

    comparison = latest_eval.merge(target, on=["dataset", "agent"], how="left")
    if "teacher_nonidle_frac" in comparison.columns and "target_teacher_nonidle_frac" in comparison.columns:
        comparison["teacher_nonidle_frac"] = comparison["target_teacher_nonidle_frac"].fillna(comparison["teacher_nonidle_frac"])
    elif "target_teacher_nonidle_frac" in comparison.columns:
        comparison["teacher_nonidle_frac"] = comparison["target_teacher_nonidle_frac"]
    comparison["pred_minus_teacher_nonidle"] = comparison["pred_nonidle_frac"] - comparison["teacher_nonidle_frac"]

    _show(
        comparison,
        columns=["checkpoint", "dataset", "agent", "epoch", "accuracy", "pred_nonidle_frac", "teacher_nonidle_frac", "pred_minus_teacher_nonidle", "false_noop_rate", "false_intervention_rate", "balanced_nonidle_frac", "nonidle_weight", "aux_weight", "soft_distillation_loss", "soft_weight", "soft_temperature"],
        sort=["checkpoint", "agent"],
    )

    fig = px.scatter(
        comparison,
        x="teacher_nonidle_frac",
        y="pred_nonidle_frac",
        color="checkpoint",
        symbol="agent",
        hover_data=[c for c in ["accuracy", "false_noop_rate", "false_intervention_rate", "pred_minus_teacher_nonidle", "soft_distillation_loss", "soft_weight"] if c in comparison.columns],
        title="Student predicted non-idle rate vs teacher dataset non-idle target",
        labels={"teacher_nonidle_frac": "teacher non-idle fraction", "pred_nonidle_frac": "student predicted non-idle fraction"},
    )
    fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line={"dash": "dash", "color": "black"})
    fig.update_layout(template="plotly_white", height=650, width=1050)
    cd.save_fig(fig, f"{SELECTION_LABEL}_student_pred_nonidle_vs_teacher_target", save=SAVE_FIGURES)
    if SHOW_FIGURES:
        fig.show()
else:
    print("Need matching student checkpoint metrics and teacher dataset summaries for this comparison.")


## Export Tables


In [ ]:
out_dir = cd.FIG_DIR / SELECTION_LABEL
out_dir.mkdir(parents=True, exist_ok=True)
exports = {
    "selected_runs.csv": selected_runs,
    "metric_availability.csv": availability,
    "survival_final.csv": survival_final_df,
    "agent_final.csv": agent_final_df,
    "joint_final.csv": joint_final,
    "diagnostic_final.csv": diagnostic_final_df,
    "full_test.csv": full_test_df,
    "teacher_datasets.csv": dataset_df,
    "teacher_dataset_agents.csv": dataset_agent_df,
    "student_checkpoints.csv": student_ckpt_df,
    "student_epochs.csv": student_epoch_df,
}
for name, table in exports.items():
    if table is not None and not table.empty:
        table.to_csv(out_dir / name, index=False)
print("wrote comparison CSVs to", out_dir)
